# Lab 6 — Build a ReAct Agent with Tool Use

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/humzaahmad906/applied-ml-academy/blob/main/content/vlm-guide/notebooks/15_lab_agents.ipynb)

Implement a ReAct (Reason + Act) agent from scratch: a Qwen2.5-1.5B-Instruct model in a Thought→Action→Observation loop with three real tools, a registry/dispatch layer, a regex parser, and guards against common failure modes.

Runs on Google Colab — for the GPU labs, set Runtime → Change runtime type → GPU.

The full write-up and stack alternatives (LangGraph, native tool-calling, MCP) live in the [lesson](../15_lab_agents.md).

In [ ]:
!pip install -q transformers accelerate torch


## The ReAct format

ReAct (Yao et al., 2022) interleaves reasoning and acting: the model emits `Thought` → `Action`, we inject an `Observation` from a real tool call, and it continues until it emits `Answer`. The model never sees a tool result until it asks for one — each Observation grounds the next Thought in a real return value, not a hallucinated one.

**Model:** `Qwen/Qwen2.5-1.5B-Instruct` (~3.1 GB bf16). CPU works at ~10 s/step; MPS or CUDA cuts that to 1-2 s.

In [ ]:
import re, math, random, datetime
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

random.seed(42); np.random.seed(42); torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"device: {device}")

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map=device
)
model.eval()


## System prompt

Small models (1.5B) need unambiguous format instructions — any vagueness produces violations that break the parser.

In [ ]:
SYSTEM_PROMPT = """\
You reason step-by-step using tools. At each turn emit EXACTLY ONE block:

  Thought: <your reasoning about what to do>
  Action: tool_name(argument)
  Answer: <your final, complete answer>

Rules:
- A Thought MUST appear before every Action.
- After an Action you will receive an Observation — use it before continuing.
- When you have all information needed, emit Answer:.
- NEVER emit Observation: yourself — the system injects that.

Available tools:
  calculator(expression)  – evaluate any Python math expression (math module in scope)
  get_datetime()          – return the current local date and time
  doc_search(query)       – search a small local knowledge base
"""


## Tools

Three tools: a sandboxed calculator (math-module only, no builtins), a datetime tool, and a tiny local knowledge base standing in for a real search index.

In [ ]:
def calculator(expression: str) -> str:
    # Strip builtins, expose only math.* — eval over user input is injection-prone.
    safe = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
    safe["__builtins__"] = {}
    try:
        return str(eval(expression, safe))  # noqa: S307
    except Exception as exc:
        return f"Error: {exc}"


def get_datetime() -> str:
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")


_KB: dict[str, str] = {
    "eiffel tower": "The Eiffel Tower, Paris. Height: 330 m (1,083 ft). Built 1889.",
    "great wall":   "The Great Wall of China stretches ~21,196 km (13,171 mi).",
    "transformer":  "Introduced in 'Attention Is All You Need' (Vaswani et al., 2017). Decoder-only variants dominate modern LLMs.",
    "python":       "Python: high-level, dynamically typed. First released 1991.",
}


def doc_search(query: str) -> str:
    q = query.lower()
    for key, val in _KB.items():
        if key in q or any(w in q for w in key.split()):
            return val
    return f"No entry found for: '{query}'"


## Tool registry and dispatch

A registry maps tool names to callables; `dispatch` guards against hallucinated tool names and wrong argument counts, returning an error string as the Observation instead of raising.

In [ ]:
REGISTRY: dict[str, callable] = {
    "calculator": calculator,
    "get_datetime": get_datetime,
    "doc_search": doc_search,
}


def dispatch(name: str, args: list[str]) -> str:
    if name not in REGISTRY:
        return f"Unknown tool '{name}'. Available: {list(REGISTRY)}"
    try:
        return REGISTRY[name](*args)
    except TypeError as exc:
        return f"Wrong arguments for '{name}': {exc}"
    except Exception as exc:
        return f"Tool '{name}' raised: {exc}"


## Parsing model output

A regex parser distinguishes three outcomes: a final `Answer:`, an `Action:` call to dispatch, or a stalled step with neither (which gets nudged on the next turn).

In [ ]:
_ACTION_RE = re.compile(r"Action:\s*(\w+)\(([^)]*)\)")
_ANSWER_RE = re.compile(r"Answer:\s*(.+)", re.DOTALL)


def _parse_args(raw: str) -> list[str]:
    if not raw.strip():
        return []
    return [a.strip().strip("\"'") for a in raw.split(",")]


def parse_step(text: str) -> tuple[str, str | None, list[str] | None]:
    """Returns ("answer", text, None) | ("action", name, args) | ("none", None, None)."""
    m = _ANSWER_RE.search(text)
    if m:
        return "answer", m.group(1).strip(), None
    m = _ACTION_RE.search(text)
    if m:
        return "action", m.group(1), _parse_args(m.group(2))
    return "none", None, None


## Generation and the agent loop

Each step does one full forward pass (greedy decoding — more stable for tool-use reasoning than sampling), parses the output, and either dispatches a tool call, returns the final answer, or nudges the model to complete the format. `max_steps` is the hard stop against infinite loops.

In [ ]:
def _generate(messages: list[dict]) -> str:
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,           # greedy — more stable for tool-use reasoning
            pad_token_id=tokenizer.eos_token_id,
        )
    new_ids = out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()


def run_agent(question: str, max_steps: int = 8, verbose: bool = True) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    for step in range(1, max_steps + 1):
        response = _generate(messages)
        if verbose:
            print(f"\n── step {step} ──────────────────────\n{response}")

        kind, payload, args = parse_step(response)

        if kind == "answer":
            return payload

        if kind == "action":
            obs = dispatch(payload, args or [])
            if verbose:
                print(f"Observation: {obs}")
            messages.append({"role": "assistant", "content": response})
            messages.append({"role": "user",      "content": f"Observation: {obs}"})
            continue

        # Stalled — no parseable step; nudge toward completing it
        messages.append({"role": "assistant", "content": response})
        messages.append({"role": "user",
                         "content": "Continue. Your next output must start with Action: or Answer:."})

    return "Agent did not reach a conclusion within the step budget."


## Try it

Run the agent on three questions that each exercise a different tool.

In [ ]:
for q in ["What is 2**10 + sqrt(144)?", "What time is it?",
          "How tall is the Eiffel Tower in feet? (1 m = 3.28084 ft)"]:
    print(f"\n{'='*50}\nQ: {q}\n{'='*50}")
    print(f">>> {run_agent(q)}\n")


## Failure modes and guards

- **Infinite loops.** `max_steps` is the primary guard. Log step count per question — systematic overrun means the system prompt needs sharpening.
- **Hallucinated tools.** Model invents `web_search`. Registry check returns an error observation naming the real tools; most instruction-tuned models self-correct on the next step.
- **Bad argument parses.** `_parse_args` splits on `,` naively. `calculator(max(1,2))` becomes `["max(1", "2)"]`. Use JSON-structured calling (see the lesson) to eliminate this class of bug in production.
- **Model emits `Observation:` itself.** Forbidden by the system prompt. If it happens, strip from `"Observation:"` onward before parsing, or add `"\nObservation"` as a stop sequence in `model.generate`.

## Stacks & alternatives

The lesson also covers production alternatives to this from-scratch loop, not run here:

- **LangGraph** — a typed `StateGraph` with explicit model/tool nodes and conditional edges; reach for it when you need branching, parallel tool calls, human-in-the-loop interrupts, or durable checkpointing.
- **Native/structured tool-calling** — Qwen2.5-Instruct accepts OpenAI-style JSON tool schemas in its chat template and emits structured `<tool_call>` blocks, eliminating regex parsing entirely.
- **MCP (Model Context Protocol)** — decouples tool providers from agent builders; expose tools once as an MCP server and any MCP-compatible client discovers and calls them.

See [15_lab_agents.md](../15_lab_agents.md) for the code and tradeoffs of each.